<a href="https://colab.research.google.com/github/Yuyang-Yao/Math_-5750_Project1/blob/main/Math_4800_3_1_simulation_2d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import scipy as sc
import numpy as np
import time

import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = (20, 8)
plt.style.use('ggplot')

from scipy.stats import bernoulli, binom, norm, uniform
from scipy.stats import lognorm

In [ ]:
def three_dim_T_N_Lrf_full_trajactory_res_Vaccario(R_core, R_inner_region, R_outer_region, D_inner, D_outer,
                                               alpha_chosen, N_num_searchers, R_start):
    '''
    will count residence time (so have to track all particles).
    Faster running for E(T_N) is in previous notebook
    media_outer= radius: (R_s, R_b)
    2015's convention
    compute t as proportion of incremet of step length for reflection boundary
    '''
    #0.1: para set up
    R_a=R_core
    R_b=R_outer_region
    R_s=R_inner_region

    D_s=D_inner
    D_b=D_outer

    #d=dimension
    alpha=alpha_chosen

    R_0=R_start

    dt_1=0.00125
    n1=20000 # 25s max

    #P1_cross =D1/(D1+D2)
    #P1_cross=D2/(D1+D2)
    T_FPT_list=[]
    Res_time_s=[]
    Res_time_b=[] # will use np.argmin(T_FPT_list) to track which particle is fastest

    # loops start for each searcher or particle
    for k in range (N_num_searchers):

        Res_time_s_k=0 # 1 is x in [-1,0)
        Res_time_b_k=0

        X_W_N = np.zeros((n1+1, 3))   # 3D, 1d: X_W_N = np.zeros(n1+1)

        #a random point on the sphere of radius r in 3D
        Z_x_0 = norm.rvs(loc=0, scale=1, size=(1,3))
        x_0 = R_0 * Z_x_0 / np.linalg.norm(Z_x_0)  # randomly sample 1 start location with radius r, uniformly on r.
        # np.linalg.norm(Z) solve norm of vector Z
        X_W_N [0] = x_0.flatten() # Because X_W_N[0] expects shape (3,), but x is (1,3).Flatten removes extra dimension so shapes match correctly.

        T_FPT=np.nan
        N_stop_1=np.nan

        dB_list_sd_norm = norm.rvs(loc=0, scale=np.sqrt(dt_1), size=(n1, 3)) # 3 coord iid, dB_list_sd_norm each is d(W(dt)), normal vector N(0, dt*I_3)
        #1d: dB_list_sd_norm=norm.rvs(loc=0, scale=np.sqrt(dt_1), size=n1)

        # full particle searching process
        for n in range(len(dB_list_sd_norm)):

            X_curr= X_W_N[n]
            dW_t_n_cur=dB_list_sd_norm[n]# vetor dW(dt), current time increment but determines next time location X(t+dt)
            X_curr_norm = np.linalg.norm(X_curr)

            if (X_curr_norm< R_s) and (X_curr_norm> R_a):
                D_curr=D_s
            elif (X_curr_norm> R_s) and (X_curr_norm< R_b):
                D_curr=D_b
            elif X_curr_norm== R_s:
                if  (np.linalg.norm(X_W_N[n-1])) < R_s and (np.linalg.norm(X_W_N[n-1])> R_a):
                    D_curr=D_s # if now on x_thr_1, if previous (n-1) in zone 1, use D1
                else:
                    D_curr=D_b

            X_new= X_curr + np.sqrt(2 * D_curr) * dW_t_n_cur# vector still work
            X_new_norm=np.linalg.norm(X_new)


            # step 2: no coin flip interface rule when cross interface, but use alpha
            if (X_curr_norm < R_s) and (X_new_norm > R_s):   # attempting Es -> Eb
                X_star=(1-alpha)*X_curr+ alpha *X_new # corss by D(X*)
                X_star_norm=np.linalg.norm(X_star)

                if X_star_norm <= R_s:
                    D_star=D_s #D_star=D(X(*)
                elif X_star_norm > R_s:
                    D_star=D_b
                X_new=X_curr + np.sqrt(2 * D_star) * dW_t_n_cur# replace previous x_new

            elif (X_curr_norm > R_s) and (X_new_norm < R_s): # attempting Eb -> Es
                X_star=(1-alpha) *X_curr+ alpha *X_new # corss by D(X*)
                X_star_norm=np.linalg.norm(X_star)

                if X_star_norm< R_s:
                    D_star=D_s #D_star=D(X(*)
                elif X_star_norm> R_s:
                    D_star=D_b
                X_new=X_curr + np.sqrt(2 * D_star) * dW_t_n_cur # replace previous x_new
                        # finish step 2


            # step 3.1: reflecting boundary at bounadry of shpere (maps back into domain)
            if X_new_norm > R_b:
                direction_dW_t_n_cur = dW_t_n_cur/ np.linalg.norm(dW_t_n_cur)
                # note use direction of increment dW_current (X_new diretion by its coordiniate points toward origin)

                X_increment_vec= np.sqrt(2 * D_b) * dW_t_n_cur # or D_curr=D_b is fine, # full attempted displacement from X_curr to X_new
                X_increment_vec_norm=np.linalg.norm(X_increment_vec)
                # Need to find t in [0,1] such that: ||X_curr + t*X_increment_vec|| = R_b
                 # coefficients of quadratic in t for ||X_curr + t*X_increment_vec|| = R_b)

                a = np.dot(X_increment_vec, X_increment_vec)      # squared length of step
                b = 2 * np.dot(X_curr, X_increment_vec)           # interaction of position and step
                c = np.dot(X_curr, X_curr) - R_b**2               # current distance^2 minus boundary^2
                disc = b**2 - 4 * a * c                          # discriminant
                disc = max(disc, 0)

                sqrt_disc = np.sqrt(disc)                        # square root of discriminant
                t1= (-b - sqrt_disc) / (2 * a)                  # first root
                t2 = (-b + sqrt_disc) / (2 * a)                  # second root

                roots = [tt for tt in (t1, t2) if 0 <= tt <= 1]
                t = min(roots) if roots else 1.0                 # choose physical root in [0,1], proportion of X_increment_vec_norm spend before touch bounadry

                # now relfection
                X_exceed_bounadry_amount= (1-t)* X_increment_vec_norm
                X_new=X_new- 2* X_exceed_bounadry_amount* direction_dW_t_n_cur
                # reflect in 1d : L_low - (X_new - L_low), t₁ = (-b − √disc)/(2a) gives the first intersection (entering boundary)

            # step 3.2: absorbing boundary at L_top
            if X_new_norm <= R_a:
                #X_new = L_top

                # IMPORTANT: store the absorbed position at index n+1 (time t_{n+1})
                X_W_N[n+1] = X_new # just ignore a bit innerward of core

            # IMPORTANT: stopping index/time correspond to n+1
                N_stop_1 = n + 1
                T_FPT  = (n + 1) * dt_1

                break

                        # normal step for updatin X_new_5: store x_{n+1}
            X_W_N[n+1] = X_new

            if X_new_norm>R_s:# X is now in media 2
                Res_time_b_k=Res_time_b_k+dt_1
            elif X_new_norm<R_s:
                Res_time_s_k=Res_time_s_k+dt_1

        if np.isnan(T_FPT):        # inital set up is T_FPT=np.nan
                                  #  this will happen if "if X_new >= L_top:" did not actually work
                                  #i.e, if current X(t) did not reach upper boundary L_top
            N_stop_1 = len(dB_list_sd_norm)
            T_FPT = N_stop_1 * dt_1



        T_FPT_list.append(T_FPT)
        Res_time_s.append(Res_time_s_k)
        Res_time_b.append(Res_time_b_k)

    T_FPT_extreme_one_X = np.min(T_FPT_list)

    index_min = np.argmin(T_FPT_list) # track particle index who is fastest
    T_FPT_extreme_one_X_Res_time_s=Res_time_s[index_min]
    T_FPT_extreme_one_X_Res_time_b=Res_time_b[index_min]
    output_dic={'extreme_FPT': T_FPT_extreme_one_X, 'residence_time_media1':T_FPT_extreme_one_X_Res_time_s,
                'residence_time_media2':T_FPT_extreme_one_X_Res_time_b }
    return output_dic


# (a) ahlpa=1, Db<Ds

In [ ]:
N_num_run=20000
Res_time_media_s_alpha_1_Vaccario=[]
Res_time_media_b_alpha_1_Vaccario=[]
FPT_alpha_1_Vaccario=[]
for m in range (N_num_run):
    output_dic_call_Vaccario_1= three_dim_T_N_Lrf_full_trajactory_res_Vaccario(0.5, 1.5, 2.5, 2, 4, 1, 1, 2.3) # output dic

    Res_time_s_m_Vaccario_1=output_dic_call_Vaccario_1['residence_time_media1']
    Res_time_b_m_Vaccario_1=output_dic_call_Vaccario_1['residence_time_media2']
    FPT_c1_m_Vaccario_1=output_dic_call_Vaccario_1['extreme_FPT']

    FPT_alpha_1_Vaccario.append(FPT_c1_m_Vaccario_1)
    Res_time_media_s_alpha_1_Vaccario.append(Res_time_s_m_Vaccario_1)
    Res_time_media_b_alpha_1_Vaccario.append(Res_time_b_m_Vaccario_1)

In [ ]:
empirical_MFPT_alpha_1_Vaccario=np.mean(FPT_alpha_1_Vaccario)
empirical_Res_time_medias_alpha_1_Vaccario=np.mean(Res_time_media_s_alpha_1_Vaccario)
empirical_Res_time_mediab_alpha_1_Vaccario=np.mean(Res_time_media_b_alpha_1_Vaccario)
print('start near outer bounadry of 3d ball,a=0.5,Rs=1.5, Rb=2.5, Ds=4, Db=2, alpha=1, Vaccario version, MFPT is'
      ,empirical_MFPT_alpha_1_Vaccario ,'s, residence time in E_{R_a<r<R_s}, alpha=1 is:',empirical_Res_time_medias_alpha_1_Vaccario,
      's, residence time in E_{R_s<r<R_b}, alpha=1 is:',empirical_Res_time_mediab_alpha_1_Vaccario,
      's. While true val is: T_s==0.2916667, T_b=1.715113, MFPT=τ=τ1​+τ2​=2.00678' )

start near outer bounadry of 3d ball,a=0.5,Rs=1.5, Rb=2.5, Ds=4, Db=2, alpha=1, Vaccario version, MFPT is 2.3628043750000005 s, residence time in E_{R_a<r<R_s}, alpha=1 is: 0.3542140624999967 s, residence time in E_{R_s<r<R_b}, alpha=1 is: 2.0073410625000174 s. While true val is: T_s==0.2916667, T_b=1.715113, MFPT=τ=τ1​+τ2​=2.00678


In [ ]:
empirical_MFPT_alpha_1_Vaccario=np.mean(FPT_alpha_1_Vaccario)
empirical_Res_time_medias_alpha_1_Vaccario=np.mean(Res_time_media_s_alpha_1_Vaccario)
empirical_Res_time_mediab_alpha_1_Vaccario=np.mean(Res_time_media_b_alpha_1_Vaccario)
print('start near outer bounadry of 3d ball,a=0.5,Rs=1.5, Rb=2.5, Ds=4, Db=2, alpha=1, Vaccario version, MFPT is'
      ,empirical_MFPT_alpha_1_Vaccario ,'s, residence time in E_{R_a<r<R_s}, alpha=1 is:',empirical_Res_time_medias_alpha_1_Vaccario,
      's, residence time in E_{R_s<r<R_b}, alpha=1 is:',empirical_Res_time_mediab_alpha_1_Vaccario,
      's. While true val is: T_s==0.2916667, T_b=1.715113, MFPT=τ=τ1​+τ2​=2.00678' )

start near outer bounadry of 3d ball,a=0.5,Rs=1.5, Rb=2.5, Ds=4, Db=2, alpha=1, Vaccario version, MFPT is 2.334553125 s, residence time in E_{R_a<r<R_s}, alpha=1 is: 0.35087524999999675 s, residence time in E_{R_s<r<R_b}, alpha=1 is: 1.9824278750000202 s. While true val is: T_s==0.2916667, T_b=1.715113, MFPT=τ=τ1​+τ2​=2.00678


In [ ]:
empirical_MFPT_alpha_1_Vaccario=np.mean(FPT_alpha_1_Vaccario)
empirical_Res_time_medias_alpha_1_Vaccario=np.mean(Res_time_media_s_alpha_1_Vaccario)
empirical_Res_time_mediab_alpha_1_Vaccario=np.mean(Res_time_media_b_alpha_1_Vaccario)
print('start near outer bounadry of 3d ball, Ds=2, Db=4, alpha=1, Vaccario version, MFPT is'
      ,empirical_MFPT_alpha_1_Vaccario ,'s, residence time in E_{R_a<r<R_s}, alpha=1 is:',empirical_Res_time_medias_alpha_1_Vaccario,
      's, residence time in E_{R_s<r<R_b}, alpha=1 is:',empirical_Res_time_mediab_alpha_1_Vaccario, 's',
     'real MFPT: 3.4808, T_s: 3.4808')

start near outer bounadry of 3d ball, Ds=2, Db=4, alpha=1, Vaccario version, MFPT is 4.005972875 s, residence time in E_{R_a<r<R_s}, alpha=1 is: 0.7070024999999956 s, residence time in E_{R_s<r<R_b}, alpha=1 is: 3.2977250000000735 s real MFPT: 3.4808, T_s: 3.4808


# (a) ahlpa=0, Db<Ds

In [ ]:
N_num_run=1000
Res_time_media_s_alpha_0_Vaccario=[]
Res_time_media_b_alpha_0_Vaccario=[]
FPT_alpha_0_Vaccario=[]
for m in range (N_num_run):
    output_dic_call_Vaccario_0= three_dim_T_N_Lrf_full_trajactory_res_Vaccario(0.5, 1.5, 2.5, 2, 4, 0, 1, 2.3) # output dic

    Res_time_s_m_Vaccario_0=output_dic_call_Vaccario_0['residence_time_media1']
    Res_time_b_m_Vaccario_0=output_dic_call_Vaccario_0['residence_time_media2']
    FPT_c1_m_Vaccario_0=output_dic_call_Vaccario_0['extreme_FPT']

    FPT_alpha_0_Vaccario.append(FPT_c1_m_Vaccario_0)
    Res_time_media_s_alpha_0_Vaccario.append(Res_time_s_m_Vaccario_0)
    Res_time_media_b_alpha_0_Vaccario.append(Res_time_b_m_Vaccario_0)

In [ ]:
empirical_MFPT_alpha_1_Vaccario=np.mean(FPT_alpha_1_Vaccario)
empirical_Res_time_medias_alpha_1_Vaccario=np.mean(Res_time_media_s_alpha_1_Vaccario)
empirical_Res_time_mediab_alpha_1_Vaccario=np.mean(Res_time_media_b_alpha_1_Vaccario)
print('start near outer bounadry of 3d ball, Ds=2, Db=4, alpha=1, Vaccario version, MFPT is'
      ,empirical_MFPT_alpha_1_Vaccario ,'s, residence time in E_{R_a<r<R_s}, alpha=1 is:',empirical_Res_time_medias_alpha_1_Vaccario,
      's, residence time in E_{R_s<r<R_b}, alpha=1 is:',empirical_Res_time_mediab_alpha_1_Vaccario, 's' )